[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S12/S12_04_embeddings_lsa.ipynb)

# S12 · 04 · Embeddings: representar el significado con vectores (LSA)

**Tiempo estimado:** 50 min · **Sprint 12** (UD 4.8) · Cliente: **TurisData Canarias**

**Lo que vas a aprender**
1. Qué es un **embedding**: un texto (o palabra) como un vector corto y *denso* en el que "cerca" significa "parecido".
2. Construir embeddings **sin internet** con **LSA** (TF-IDF + `TruncatedSVD`).
3. Medir parecido con la **similitud del coseno** y buscar reseñas parecidas.
4. Usar los embeddings como entrada de un clasificador y compararlos con TF-IDF.
5. (Opcional, con internet) Cambiar LSA por un modelo neuronal de `sentence-transformers`.

**Contexto.** TurisData quiere, dada una queja, encontrar rápidamente reseñas parecidas para ver si es un problema recurrente.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

resenas = pd.read_csv(dato("resenas.csv"))

# Palabras vacías y de "relleno" (frases que abren muchas reseñas sin decir nada de la estancia). No quitamos «no».
VACIAS = {"a", "al", "con", "de", "del", "el", "en", "es", "esta", "estaba", "era", "fue", "la", "las", "lo", "los", "me", "mi",
          "muy", "nos", "para", "por", "que", "se", "su", "un", "una", "y", "o", "pasamos", "semana", "aquí", "escapada", "fin",
          "viajamos", "familia", "fuimos", "trabajo", "general", "todo", "estancia", "sin", "más"}

def limpiar(texto):
    palabras = re.sub(r"[^a-záéíóúüñ\s]", " ", texto.lower()).split()
    return " ".join(p for p in palabras if p not in VACIAS)

## 1. Idea: de la bolsa de palabras al vector denso
TF-IDF da a cada reseña un vector **enorme y casi vacío** (una columna por palabra, casi todo ceros).
Dos reseñas que hablan de lo mismo con palabras distintas ("ruidoso" / "no nos dejó dormir") quedan *lejos*.

Un **embedding** comprime eso a pocas decenas de números donde las dimensiones capturan **temas** que van juntos.
**LSA** (*Latent Semantic Analysis*) lo hace con una descomposición matemática (SVD) que busca las combinaciones de
palabras que más varían; se la puede imaginar como "descubrir los ejes de conversación" del corpus.

In [ ]:
tfidf = TfidfVectorizer(preprocessor=limpiar, min_df=2)
M = tfidf.fit_transform(resenas["texto"])
print("TF-IDF:", M.shape, "→ proporción de ceros:", round(1 - M.nnz / (M.shape[0] * M.shape[1]), 3))

**Ejercicio 1.** Reduce `M` a **12 dimensiones** con `TruncatedSVD(n_components=12, random_state=SEMILLA)`.
Guarda el modelo en `lsa` y los vectores de las reseñas en `Z` (forma `(1500, 12)`).
Guarda también en `var_explicada` la suma de la varianza explicada (`lsa.explained_variance_ratio_.sum()`).

In [ ]:
# TODO: lsa = TruncatedSVD(...); Z = lsa.fit_transform(M)
...
print("Forma de Z:", Z.shape, "| varianza explicada con 12 ejes:", round(var_explicada, 3))
assert Z.shape == (len(resenas), 12), "Z debe tener 12 columnas: una por dimensión de LSA."
assert 0.2 < var_explicada < 1, "var_explicada debe ser un número entre 0 y 1 (suma del ratio de varianza)."

## 2. ¿Qué "significa" cada eje? Palabras que van juntas
Cada eje de LSA es una combinación de palabras. Mirando las palabras con más peso en los primeros ejes
vemos qué "temas" ha encontrado **sin que le digamos nada**.

In [ ]:
terminos = np.array(tfidf.get_feature_names_out())
for i in range(5):
    peso = lsa.components_[i]
    arriba = terminos[np.argsort(peso)[-6:][::-1]]
    print(f"Eje {i + 1}: {', '.join(arriba)}")

## 3. Similitud del coseno
Para medir "cuánto se parecen" dos vectores se usa el **coseno del ángulo** entre ellos:
1 = misma dirección (muy parecidos), 0 = sin relación. No depende de la longitud del texto.

In [ ]:
a, b, c = np.array([1, 0]), np.array([1, 1]), np.array([0, 1])
print("coseno(a, b) =", round(float(cosine_similarity([a], [b])[0, 0]), 3), "(45°)")
print("coseno(a, c) =", round(float(cosine_similarity([a], [c])[0, 0]), 3), "(90°, nada en común)")

**Ejercicio 2.** Completa `parecidas(i, k=3)`: devuelve los índices de las `k` reseñas más parecidas (por coseno sobre `Z`)
a la reseña número `i`, **excluyendo la propia `i`**, de más a menos parecida.

In [ ]:
def parecidas(i, k=3):
    # TODO: cosine_similarity([Z[i]], Z)[0], poner -1 en la posición i y usar argsort descendente
    ...

vecinas = parecidas(0, 3)
print("Reseña 0:", resenas.loc[0, "texto"])
for j in vecinas:
    print("  ≈", resenas.loc[j, "texto"])
assert len(vecinas) == 3 and 0 not in vecinas, "parecidas debe devolver k índices distintos de i."
sim0 = cosine_similarity([Z[0]], Z)[0]
assert sim0[vecinas[0]] >= sim0[vecinas[1]] >= sim0[vecinas[2]], "Ordena de más a menos parecida."

## 4. Mapa de reseñas en 2 dimensiones
Con solo 2 ejes podemos dibujar el corpus. Coloreamos por sentimiento (que LSA **no ha visto**).

In [ ]:
Z2 = TruncatedSVD(n_components=2, random_state=SEMILLA).fit_transform(M)
colores = {"positivo": "#3b8f5a", "neutro": "#999999", "negativo": "#c0463a"}
fig, ax = plt.subplots()
for s, col in colores.items():
    m = (resenas["sentimiento"] == s).values
    ax.scatter(Z2[m, 0], Z2[m, 1], s=12, alpha=.5, c=col, label=s)
ax.set_title("Reseñas en 2 dimensiones (LSA)"); ax.set_xlabel("Eje 1"); ax.set_ylabel("Eje 2"); ax.legend()
plt.show()

## 5. Embeddings como entrada de un clasificador
Comparamos: regresión logística con **TF-IDF completo** frente a solo **12 números por reseña (LSA)**.
Ojo: LSA es un modelo que se ajusta con datos; para no filtrar información, se ajusta **solo con el entrenamiento**.

**Ejercicio 3.** Parte los datos en entrenamiento y prueba (ya hecho abajo). Ajusta `tfidf_e` y `lsa_e` **solo con `X_ent`**, entrena una
regresión logística sobre los vectores LSA de entrenamiento y guarda en `acc_lsa` la exactitud en prueba.

In [ ]:
X_ent, X_test, y_ent, y_test = train_test_split(resenas["texto"], resenas["sentimiento"], test_size=0.25,
                                                stratify=resenas["sentimiento"], random_state=SEMILLA)
tfidf_e = TfidfVectorizer(preprocessor=limpiar, min_df=2)
# TODO: fit_transform en entrenamiento y transform en prueba, tanto para tfidf_e como para lsa_e (12 componentes)
...
clf_tfidf = LogisticRegression(max_iter=1000).fit(Me_ent, y_ent)
acc_tfidf = accuracy_score(y_test, clf_tfidf.predict(Me_test))
print(f"TF-IDF completo ({Me_ent.shape[1]} columnas): {acc_tfidf:.3f}")
print(f"LSA (12 columnas)                : {acc_lsa:.3f}")
assert Ze_ent.shape[0] == len(X_ent) and Ze_test.shape[0] == len(X_test), "Ajusta con X_ent y transforma X_test."
assert acc_lsa > 0.6, "acc_lsa debería superar 0.6; revisa que entrenes con Ze_ent/y_ent y evalúes con Ze_test."

## 6. Mini-reto integrador: buscador semántico de reseñas
Crea `buscar(consulta, k=3)`: convierte la consulta al espacio LSA (`tfidf.transform` y `lsa.transform`; usa `limpiar` como ya está en `tfidf`),
calcula el coseno con `Z` y devuelve los `k` textos más parecidos. Prueba con `"no me dejó dormir el ruido"`.

In [ ]:
def buscar(consulta, k=3):
    # TODO: transforma la consulta con tfidf y luego con lsa; compara con Z y ordena de mayor a menor
    ...

hallazgos = buscar("no me dejó dormir el ruido")
for h in hallazgos:
    print("-", h)
assert len(hallazgos) == 3, "buscar debe devolver k textos."
assert any(("ruid" in h.lower() or "dormir" in h.lower() or "silencio" in h.lower() or "oye" in h.lower()) for h in hallazgos), \
    "Los resultados deberían hablar de ruido/dormir. Revisa que uses lsa.transform sobre la consulta y compares con Z."

## 7. OPCIONAL · Embeddings neuronales con `sentence-transformers` (requiere internet)
> **⚠️ NO VERIFICADO EN EL TALLER:** esta parte necesita descargar un modelo (~120 MB) desde Hugging Face y instalar una librería.
> Ejecútala en **Google Colab** o en casa con conexión. Si no tienes internet, sáltala: el resto del notebook funciona sin ella.

La diferencia con LSA: un modelo neuronal ya *entrenado con millones de frases* entiende que "ruidoso" y "no pude dormir" se parecen
aunque en tu corpus casi no coincidan palabras. LSA solo conoce **tu** corpus.

In [ ]:
USAR_SENTENCE_TRANSFORMERS = False   # ← cámbialo a True solo si tienes internet (Colab)

if USAR_SENTENCE_TRANSFORMERS:
    # NO VERIFICADO: requiere internet
    %pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer
    modelo_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    E = modelo_st.encode(resenas["texto"].tolist(), show_progress_bar=False, normalize_embeddings=True)
    q = modelo_st.encode(["no me dejó dormir el ruido"], normalize_embeddings=True)
    mejores = np.argsort((E @ q.T).ravel())[::-1][:3]
    for i in mejores:
        print("-", resenas.loc[i, "texto"])
else:
    print("Parte opcional omitida (USAR_SENTENCE_TRANSFORMERS = False).")

## Preguntas de reflexión
1. ¿Por qué los vectores LSA son "densos" y TF-IDF "dispersos"? ¿Qué ganamos y qué perdemos al comprimir de miles a 12 números?
2. Si hubiéramos ajustado el LSA con todas las reseñas antes de partir en train/test, ¿qué tipo de error cometeríamos?
3. Un embedding preentrenado (sentence-transformers) conoce el español general; ¿qué ventaja y qué riesgo tiene frente a LSA sobre nuestro corpus?
4. ¿Qué problemas de privacidad hay al enviar reseñas a un servicio externo para calcular embeddings?

## Resumen: qué has aprendido
- Un **embedding** representa un texto como un vector corto; "cerca" (coseno alto) = "parecido".
- **LSA** crea embeddings offline con TF-IDF + SVD; útil, rápido y explicable.
- Los embeddings sirven para **buscar por parecido** y como entrada de clasificadores.
- Los modelos neuronales preentrenados captan más significado, a costa de descargas y de depender de un tercero.